In [1]:
import pandas as pd


In [13]:
# Load residual series and parse index as dates
temp_resid = pd.read_csv(
    "../data/deseasonalised/temp_resid.csv",
    index_col=0,
    parse_dates=True
)

price_resid = pd.read_csv(
    "../data/deseasonalised/price_resid.csv",
    index_col=0,
    parse_dates=True
)

# Extract the deseasonalised residual series
temp_res = temp_resid["temp_deseasoned"].copy()
price_res = price_resid["price_deseasoned"].copy()

# Ensure the indexes are datetime indexes
temp_res.index = pd.to_datetime(temp_res.index)
price_res.index = pd.to_datetime(price_res.index)

# Sort both series by time
temp_res = temp_res.sort_index()
price_res = price_res.sort_index()

# Give explicit names to the series
temp_res.name = "temp"
price_res.name = "price"

# Align both series on their common dates only
Y = pd.concat(
    [temp_res, price_res],
    axis=1,
    join="inner"
).dropna()

# Inspect the aligned dataset
print("Temperature range:", temp_res.index.min(), "->", temp_res.index.max(), "| n =", len(temp_res))
print("Price range:", price_res.index.min(), "->", price_res.index.max(), "| n =", len(price_res))
print("Aligned range:", Y.index.min(), "->", Y.index.max(), "| n =", len(Y))
print(Y.head())
print(Y.tail())

# Convert to NumPy for statsmodels / custom likelihood code
Y_np = Y.to_numpy()

# Keep the aligned time index
time_index = Y.index.copy()

Temperature range: 2020-01-01 00:00:00+00:00 -> 2025-12-31 23:00:00+00:00 | n = 52608
Price range: 2023-01-01 00:00:00+00:00 -> 2025-12-31 00:00:00+00:00 | n = 26281
Aligned range: 2023-01-01 00:00:00+00:00 -> 2025-12-31 00:00:00+00:00 | n = 26281
                                temp     price
datetime                                      
2023-01-01 00:00:00+00:00  13.933978 -0.026323
2023-01-01 01:00:00+00:00  14.317873 -0.024259
2023-01-01 02:00:00+00:00  14.389412 -0.024685
2023-01-01 03:00:00+00:00  14.607491 -0.023113
2023-01-01 04:00:00+00:00  14.779772 -0.024713
                               temp     price
datetime                                     
2025-12-30 20:00:00+00:00 -3.206245 -0.008414
2025-12-30 21:00:00+00:00 -3.411489 -0.005823
2025-12-30 22:00:00+00:00 -3.478863 -0.001588
2025-12-30 23:00:00+00:00 -3.678123  0.003609
2025-12-31 00:00:00+00:00 -3.772607  0.008535


In [14]:
from statsmodels.tsa.api import VAR

# Fit a VAR(1) model without deterministic trend
var_model = VAR(Y)
var_res = var_model.fit(10, trend="n")

print(var_res.summary())

# Extract the AR coefficient matrix
Phi1 = var_res.coefs[0]

# Extract the innovation covariance matrix
Sigma_u = var_res.sigma_u

print("Phi1:")
print(Phi1)

print("Innovation covariance:")
print(Sigma_u)

  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Sat, 09, May, 2026
Time:                     17:25:13
--------------------------------------------------------------------
No. of Equations:         2.00000    BIC:                   -9.87615
Nobs:                     26271.0    HQIC:                  -9.88458
Log likelihood:           55377.8    FPE:                5.07500e-05
AIC:                     -9.88860    Det(Omega_mle):     5.06729e-05
--------------------------------------------------------------------
Results for equation temp
               coefficient       std. error           t-stat            prob
----------------------------------------------------------------------------
L1.temp           1.269490         0.006169          205.773           0.000
L1.price         -2.297623         0.293302           -7.834           0.000
L2.temp          -0.252300         0.009975          -25.294           0.000
